# ZK proof lifecycle

This notebook makes the dual-anchor redaction pipeline legible: commit a document,
generate a selective-disclosure proof, anchor the Poseidon root in the BLAKE3 SMT,
and finally call `verify_zk_redaction()` to show the public inputs Olympus would hand
to `snarkjs groth16 verify`.


In [ ]:
from unittest.mock import patch

from protocol.hashes import blake3_to_field_element
from protocol.poseidon_bn128 import poseidon_hash_bn128
from protocol.poseidon_tree import PoseidonMerkleTree
from protocol.redaction import RedactionProtocol
from protocol.redaction_ledger import verify_zk_redaction
from protocol.ssmf import SparseMerkleTree


In [1]:
document_parts = [
    'Executive summary',
    'Budget appendix',
    'Names redacted under statute',
]
revealed_indices = [0, 2]

tree, blake3_root = RedactionProtocol.commit_document(document_parts)
proof = RedactionProtocol.create_redaction_proof(tree, revealed_indices)
print(f'BLAKE3 commitment root: {blake3_root}')
print(f'Revealed indices: {revealed_indices}')
print(f'Revealed hashes: {proof.revealed_hashes}')
print(f'Merkle proof sibling counts: {[len(p.siblings) for p in proof.merkle_proofs]}')
print(
    'verify_redaction_proof(...) =>',
    RedactionProtocol.verify_redaction_proof(proof, [document_parts[0], document_parts[2]]),
)


BLAKE3 commitment root: 7b4bafde77d0237b5897ee55c65ac498b56fe9a2b9c26f5c3dee45b00ccbe63f
Revealed indices: [0, 2]
Revealed hashes: ['8d6565c8fefc4ac7243611ef8d04942b3b7cadfff6ab5a77a607d184ee9d91b2', '840355c3d562a92ca3f17193459bf4ba3a66ab0c52948828f752ee4a1828a916']
Merkle proof sibling counts: [2, 2]
verify_redaction_proof(...) => True


In [2]:
leaf_fields = [int(blake3_to_field_element(part.encode('utf-8'))) for part in document_parts]
padded_leaves = leaf_fields + [0] * (16 - len(leaf_fields))
poseidon_tree = PoseidonMerkleTree(padded_leaves, depth=4)
poseidon_root = poseidon_tree.get_root()

reveal_mask = [1 if i in revealed_indices else 0 for i in range(16)]
revealed_count = sum(reveal_mask)
revealed_leaves = [leaf if mask else 0 for leaf, mask in zip(padded_leaves, reveal_mask)]

acc = poseidon_hash_bn128(revealed_count, revealed_leaves[0])
for leaf in revealed_leaves[1:]:
    acc = poseidon_hash_bn128(acc, leaf)
redacted_commitment = str(acc)

smt = SparseMerkleTree()
tree, dual_blake3_root = RedactionProtocol.commit_document_dual(
    document_parts=document_parts,
    poseidon_root=poseidon_root,
    smt=smt,
    document_id='civic-budget',
    version=1,
)
ledger_bundle = RedactionProtocol.create_redaction_proof_with_ledger(
    tree=tree,
    revealed_indices=revealed_indices,
    poseidon_root=poseidon_root,
    smt=smt,
    document_id='civic-budget',
    version=1,
    zk_proof={'pi_a': ['1', '2'], 'pi_b': [['3', '4'], ['5', '6']], 'pi_c': ['7', '8']},
    redacted_commitment=redacted_commitment,
    revealed_count=revealed_count,
)

print(f'Poseidon leaf field elements (unpadded): {leaf_fields}')
print(f'Poseidon root (depth=4 padded tree): {poseidon_root}')
print(f'revealMask: {reveal_mask}')
print(f'revealedCount: {revealed_count}')
print(f'redactedCommitment: {redacted_commitment}')
print(f'SMT root: {smt.get_root().hex()}')
print('verify_smt_anchor(...) =>', ledger_bundle.verify_smt_anchor(smt.get_root()))


Poseidon leaf field elements (unpadded): [20178779955328095444875476211327044074961021240177980632126859239940199125424, 9734317475366685819443070739365105346898506481817261756238475635524546489494, 15934702738608130105870299980688507710237798145397315026108401949272594491668]
Poseidon root (depth=4 padded tree): 13048633151337393974142394679515379509089824099038622246241209950111627800733
revealMask: [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
revealedCount: 2
redactedCommitment: 6400773959011045218274160748052284103547707247409641813609176118773497714394
SMT root: 8d2aa1311cabd8e0448a35ee32acf3afcf10b2ef7fda6072f19e6686fef512e6
verify_smt_anchor(...) => True


In [3]:
with patch('protocol.redaction_ledger.Groth16Prover.verify', return_value=True) as mock_verify:
    zk_result = verify_zk_redaction(ledger_bundle.zk_proof, ledger_bundle.zk_public_inputs)
    forwarded_public_signals = mock_verify.call_args.kwargs['proof'].public_signals

print('verify_zk_redaction(...) =>', zk_result)
print('Groth16Prover.verify received public signals:', forwarded_public_signals)
print(
    (
        'Notebook note: this cell patches Groth16Prover.verify because the repository does not ship '
        'a redaction_validity verification key or proof artifact. The point here is to expose the '
        'exact public input mapping performed by verify_zk_redaction().'
    )
)


verify_zk_redaction(...) => True
Groth16Prover.verify received public signals: ['13048633151337393974142394679515379509089824099038622246241209950111627800733', '6400773959011045218274160748052284103547707247409641813609176118773497714394', '2']
Notebook note: this cell patches Groth16Prover.verify because the repository does not ship
a redaction_validity verification key or proof artifact. The point here is to expose the
exact public input mapping performed by verify_zk_redaction().
